# 01 — Data exploration (fault-free TEP data)

**Production Pulse — Phase 2.** Loads and inspects the two **fault-free** datasets only, using the project modules in `src/production_pulse`.

- Original `.RData` files are opened **read-only**; nothing is written to them.
- The large faulty files are **not** loaded here (the loader's size guard would refuse them anyway).
- This notebook has **not been run yet** — outputs appear only after you run it on Windows with the `.venv313` kernel.
- Write what you observe into `docs/RESEARCH_LOGBOOK.md` (last cell has prompts).

## 1. Setup — locate the project and import the modules

In [ ]:
import sys
from pathlib import Path

# Find the project root by walking up from the notebook's folder (no hard-coded username).
_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_here, *_here.parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "production_pulse").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the project root. Open this notebook from inside the project folder.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd

from production_pulse import get_paths, load_config, setup_logging
from production_pulse.data import (
    FEATURE_COLUMNS, METADATA_COLUMNS, XMEAS_COLUMNS, XMV_COLUMNS,
    load_raw_dataset, split_metadata_features, summarize_tep_frame,
)
from production_pulse.preprocessing import FeatureScaler
from production_pulse.splitting import apply_split, check_frames_no_overlap, split_runs

paths = get_paths()
config = load_config(root=paths.root)
logger = setup_logging(config=config, paths=paths)   # log file: logs/production_pulse.log

print("Python      :", sys.executable)
print("Project root:", paths.root)
print("numpy", np.__version__, "| pandas", pd.__version__)
print("Running in .venv313:", Path(sys.prefix).name.lower() == ".venv313")

## 2. Load the fault-free TRAINING file (validated on load)

In [ ]:
train_raw = load_raw_dataset("fault_free_training", paths=paths, config=config)
train_summary = summarize_tep_frame(train_raw)
print(train_summary.format())

## 3. Load the fault-free TESTING file

In [ ]:
test_raw = load_raw_dataset("fault_free_testing", paths=paths, config=config)
test_summary = summarize_tep_frame(test_raw)
print(test_summary.format())

## 4. Inspect — do not assume
Check what the files actually contain (column agreement, `faultNumber` values, dtypes, first rows) before relying on any documented TEP convention.

In [ ]:
print("Same columns in both files:", list(train_raw.columns) == list(test_raw.columns))
print("faultNumber values  train:", train_summary.fault_numbers, "| test:", test_summary.fault_numbers)
print("Column dtypes (train):", train_summary.dtype_counts)
train_raw.head()

## 5. Separate metadata from the 52 measurement features

In [ ]:
train_meta, train_x = split_metadata_features(train_raw)
print("metadata columns:", list(train_meta.columns))
print("feature columns :", len(train_x.columns), "->", train_x.columns[0], "...", train_x.columns[-1])
train_x.describe().T.head(15)

## 6. Basic plots
Needs **matplotlib**, which was *not* found in `.venv313` when Claude checked the installed packages. If the cell below says it is missing, tell Claude before installing anything. Figures are only saved if you set `SAVE_FIGURES = True` (they go to `reports/figures/`, never next to the raw data).

In [ ]:
SAVE_FIGURES = False

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed in this environment - plots skipped (see note above).")

def save(fig, name):
    if SAVE_FIGURES:
        from production_pulse.paths import safe_output_path
        target = safe_output_path(paths.figures_dir / name, paths, overwrite=True)
        fig.savefig(target, dpi=150, bbox_inches="tight")
        print("saved", target)

In [ ]:
# 6a. A few sensors over time for one simulation run (training file)
if plt is not None:
    RUN = int(train_raw["simulationRun"].min())
    run_df = train_raw[train_raw["simulationRun"] == RUN]
    cols = ["xmeas_1", "xmeas_7", "xmeas_9", "xmv_10"]
    fig, axes = plt.subplots(len(cols), 1, figsize=(10, 2.2 * len(cols)), sharex=True)
    for ax, c in zip(axes, cols):
        ax.plot(run_df["sample"], run_df[c], lw=0.8)
        ax.set_ylabel(c)
    axes[-1].set_xlabel("sample")
    fig.suptitle(f"Fault-free training data, simulationRun {RUN}")
    fig.tight_layout()
    save(fig, "eda_run_timeseries.png")
    plt.show()

In [ ]:
# 6b. Distributions of a few features: training vs testing file
if plt is not None:
    cols = ["xmeas_1", "xmeas_7", "xmeas_9", "xmv_10"]
    fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3))
    for ax, c in zip(axes, cols):
        ax.hist(train_raw[c], bins=60, alpha=0.6, density=True, label="train")
        ax.hist(test_raw[c], bins=60, alpha=0.6, density=True, label="test")
        ax.set_title(c)
    axes[0].legend()
    fig.tight_layout()
    save(fig, "eda_feature_histograms.png")
    plt.show()

In [ ]:
# 6c. Correlation matrix of the 52 features (training file)
if plt is not None:
    corr = train_x.corr().to_numpy()
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(0, 52, 4)); ax.set_xticklabels(list(FEATURE_COLUMNS)[::4], rotation=90, fontsize=7)
    ax.set_yticks(range(0, 52, 4)); ax.set_yticklabels(list(FEATURE_COLUMNS)[::4], fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
    ax.set_title("Feature correlations - fault-free training")
    fig.tight_layout()
    save(fig, "eda_correlation_matrix.png")
    plt.show()

## 7. Sanity check: run-level split + training-only scaling
Splits the **training file** by `simulationRun` (seed from `configs/default.toml`) and fits the scaler on the train part only. Nothing is saved; no model is trained.

In [ ]:
split = split_runs(train_raw, config.split.fractions(), seed=config.random_seed,
                   group_columns=config.split.group_columns)
parts = apply_split(train_raw, split)
check_frames_no_overlap(parts, config.split.group_columns)
print({name: f"{len(split.runs(name))} runs / {len(df):,} rows" for name, df in parts.items()})
print("First validation runs:", split.runs("validation")[:10])

scaler = FeatureScaler().fit(parts["train"], source="fault_free_training/train split")
x_train = scaler.transform(parts["train"])
x_val = scaler.transform(parts["validation"])        # transform only - no refit
print("train features: mean %.4f, std %.4f" % (x_train.mean(), x_train.std()))
print("val   features: mean %.4f, std %.4f" % (x_val.mean(), x_val.std()))
print("constant features in train split:", scaler.constant_features_)

## 8. Notes for the research logbook
After running, record in `docs/RESEARCH_LOGBOOK.md`:
- Did the shapes, run counts and samples-per-run match the Phase 1 audit?
- Any duplicate rows/keys, constant features or surprising distributions?
- Do training and testing distributions look alike (6b)?
- Which variables are strongly correlated (6c)? These matter for the anomaly-injection design in Phase 3.